In [9]:
# ============================================================
# ML-DRIVEN ARPU OPTIMIZATION
# ============================================================

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [1]:
import sys
from pathlib import Path
import importlib.util

# Add src directory to Python path
src_path = Path.cwd().parent / 'src'
sys.path.insert(0, str(src_path))

# Import custom_transformers module properly
import custom_transformers

# Verify the classes are in the module
print("Classes in custom_transformers module:")
print(f"  - DropColumns: {hasattr(custom_transformers, 'DropColumns')}")
print(f"  - DateFeatures: {hasattr(custom_transformers, 'DateFeatures')}")
print(f"  - CyclicalFeatures: {hasattr(custom_transformers, 'CyclicalFeatures')}")

# Make sure the module is registered in sys.modules
sys.modules['custom_transformers'] = custom_transformers

# Also register the classes in __main__ just in case
import __main__
__main__.DropColumns = custom_transformers.DropColumns
__main__.DateFeatures = custom_transformers.DateFeatures
__main__.CyclicalFeatures = custom_transformers.CyclicalFeatures

# Now load the model
import joblib

model_path = Path('../models/mobile_data_consumption_pipeline.pkl')
if model_path.exists():
    print(f"\nLoading model from: {model_path}")
    model_pipeline = joblib.load(model_path)
    print("✓ Model loaded successfully!")
    print(f"Model type: {type(model_pipeline)}")
    
    # Show pipeline structure
    if hasattr(model_pipeline, 'named_steps'):
        print("\nPipeline steps:")
        for step_name, step in model_pipeline.named_steps.items():
            print(f"  • {step_name}: {type(step).__name__}")
else:
    print(f"\nModel not found at: {model_path}")
    print(f"Current working directory: {Path.cwd()}")
    print(f"Looking for: {model_path.resolve()}")

Classes in custom_transformers module:
  - DropColumns: True
  - DateFeatures: True
  - CyclicalFeatures: True

Loading model from: ..\models\mobile_data_consumption_pipeline.pkl
✓ Model loaded successfully!
Model type: <class 'sklearn.pipeline.Pipeline'>

Pipeline steps:
  • preprocessing: Pipeline
  • model: GradientBoostingRegressor


In [7]:
# ============================================================
# LOAD TEST DATA
# ============================================================

TEST_DATA_PATH = "../data/processed/test_data.parquet"

test_data = pd.read_parquet(TEST_DATA_PATH)

print(f"Test data shape: {test_data.shape}")

display(test_data.head())

Test data shape: (2002, 18)


,user_id,measurement_date,hours_streaming,hours_social,hours_messaging,hours_gaming,is_peak_hour_user,is_weekend,age_group,plan_type,device_type,network_type,data_usage_category,streaming_data_gb,social_data_gb,messaging_data_gb,gaming_data_gb,total_data_gb
0,SUB8984705,2026-03-31,1.73,4.73,1.23,0.85,1,1,25-34,Postpaid_Premium,Premium_Smartphone,4G,Moderate,2.16688,1.38049,0.01361,0.11156,4.13474
1,SUB6140709,2026-03-20,0.40,0.26,0.17,1.82,0,0,55+,Prepaid_Monthly,Mid_Range,4G+,Light,0.55106,0.07290,0.00169,0.28788,1.00957
2,SUB3654810,2026-04-16,0.91,0.28,0.01,0.03,0,0,55+,Prepaid_Daily,Mid_Range,4G,Light,1.00330,0.04974,0.00014,0.00196,1.04403
3,SUB6876924,2026-04-01,1.66,1.18,0.09,0.25,0,0,18-24,Postpaid_Premium,Basic_Phone,4G+,Moderate,3.94730,0.29067,0.00179,0.02685,3.88007
4,SUB3498002,2026-03-27,0.45,1.87,0.05,0.63,0,0,55+,Postpaid_Unlimited,Premium_Smartphone,4G,Light,0.78657,0.36445,0.00097,0.12166,1.33496


In [4]:
# ============================================================
# GENERATE PREDICTIONS
# ============================================================

TARGET = "total_data_gb"

X_test = test_data.drop(columns=[TARGET])
y_test = test_data[TARGET]

test_data["predicted_data_gb"] = model_pipeline.predict(X_test)

print("Predictions generated successfully.")

display(
    test_data[
        [
            TARGET,
            "predicted_data_gb"
        ]
    ].head(10)
)

Predictions generated successfully.


,total_data_gb,predicted_data_gb
0,4.13474,3.915709
1,1.00957,0.832581
2,1.04403,1.136447
3,3.88007,4.646118
4,1.33496,1.121036
5,3.89173,4.505812
6,1.59625,1.444136
7,0.25320,0.266354
8,2.13985,2.833531
9,0.54184,0.709489


In [11]:
# ============================================================
# FINAL MODEL PERFORMANCE
# ============================================================

if "predicted_data_gb" not in test_data.columns:
    test_data["predicted_data_gb"] = model_pipeline.predict(X_test)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_data["predicted_data_gb"]
    )
)

mae = mean_absolute_error(
    y_test,
    test_data["predicted_data_gb"]
)

r2 = r2_score(
    y_test,
    test_data["predicted_data_gb"]
)

print("=" * 70)
print("FINAL MODEL PERFORMANCE ON UNSEEN TEST DATA")
print("=" * 70)

print(f"MAE : {mae:.4f} GB")
print(f"RMSE: {rmse:.4f} GB")
print(f"R²  : {r2:.4f}")

FINAL MODEL PERFORMANCE ON UNSEEN TEST DATA
MAE : 0.4573 GB
RMSE: 0.9223 GB
R²  : 0.9562


In [12]:
# ============================================================
# BUSINESS KPI: MAE THRESHOLD
# ============================================================

MAE_TARGET = 1.5

print("\nML BUSINESS KPI")
print("-" * 30)

if mae <= MAE_TARGET:
    print(
        f"PASS: MAE = {mae:.2f} GB "
        f"(Target ≤ {MAE_TARGET} GB)"
    )
else:
    print(
        f"FAIL: MAE = {mae:.2f} GB "
        f"(Target ≤ {MAE_TARGET} GB)"
    )


ML BUSINESS KPI
------------------------------
PASS: MAE = 0.46 GB (Target ≤ 1.5 GB)


In [13]:
# ============================================================
# PREDICTION ERROR
# ============================================================

test_data["prediction_error_gb"] = (
    test_data["total_data_gb"]
    - test_data["predicted_data_gb"]
)

test_data["absolute_error_gb"] = (
    test_data["prediction_error_gb"].abs()
)

display(
    test_data[
        [
            "total_data_gb",
            "predicted_data_gb",
            "prediction_error_gb",
            "absolute_error_gb"
        ]
    ].head(10)
)

,total_data_gb,predicted_data_gb,prediction_error_gb,absolute_error_gb
0,4.13,3.92,0.22,0.22
1,1.01,0.83,0.18,0.18
2,1.04,1.14,-0.09,0.09
3,3.88,4.65,-0.77,0.77
4,1.33,1.12,0.21,0.21
5,3.89,4.51,-0.61,0.61
6,1.60,1.44,0.15,0.15
7,0.25,0.27,-0.01,0.01
8,2.14,2.83,-0.69,0.69
9,0.54,0.71,-0.17,0.17


In [14]:
# ============================================================
# CHECK BUSINESS VARIABLES
# ============================================================

business_columns = [
    "arpu_zar",
    "plan_type",
    "age_group",
    "device_type",
    "network_type",
    "is_peak_hour_user"
]

available_columns = [
    col for col in business_columns
    if col in test_data.columns
]

print("Available business columns:")
print(available_columns)

Available business columns:
['plan_type', 'age_group', 'device_type', 'network_type', 'is_peak_hour_user']


In [19]:
PRICING_BENCHMARKS = {
    'Postpaid': {
        'base_rate_per_gb': 49.00,  # R49/GB for postpaid
        'monthly_fee': 99.00,       # Base monthly fee
        'premium_multiplier': 1.3,
        'entry_multiplier': 0.7
    },
    'Prepaid': {
        'base_rate_per_gb': 69.00,  # R69/GB for prepaid (higher per GB)
        'monthly_fee': 29.00,       # Minimum top-up
        'premium_multiplier': 1.2,
        'entry_multiplier': 0.65
    },
    'Prepaid_Daily': {
        'base_rate_per_gb': 99.00,  # R99/GB for daily bundles
        'monthly_fee': 15.00,       # Daily rate
        'premium_multiplier': 1.1,
        'entry_multiplier': 0.6
    },
    'Postpaid_Basic': {
        'base_rate_per_gb': 59.00,  # R59/GB for basic postpaid
        'monthly_fee': 79.00,       # Basic monthly fee
        'premium_multiplier': 1.25,
        'entry_multiplier': 0.65
    }
}

def calculate_arpu_zar(row):
    """Calculate ARPU in South African Rands based on usage and plan type"""
    plan = row['plan_type']
    usage_gb = row['total_data_gb']
    
    # Get pricing for this plan type
    pricing = PRICING_BENCHMARKS.get(plan, PRICING_BENCHMARKS['Prepaid'])
    
    # Base ARPU calculation (data usage + monthly fee)
    data_charge = usage_gb * pricing['base_rate_per_gb']
    monthly_fee = pricing['monthly_fee']
    
    # Apply volume discounts based on usage tiers
    if usage_gb > 30:
        # Heavy users get volume discount (MTN/Vodacom do this)
        arpu = (data_charge * 0.75) + monthly_fee
    elif usage_gb > 20:
        arpu = (data_charge * 0.82) + monthly_fee
    elif usage_gb > 10:
        arpu = (data_charge * 0.90) + monthly_fee
    elif usage_gb > 5:
        arpu = data_charge + monthly_fee
    else:
        # Light users pay premium per GB (typical for prepaid)
        arpu = (data_charge * 1.15) + monthly_fee
    
    # Ensure minimum ARPU (realistic SA market floors)
    min_arpu = 29 if 'Prepaid' in plan else 79
    return round(max(min_arpu, arpu), 2)

# Calculate ARPU for each user
test_data['arpu_zar'] = test_data.apply(calculate_arpu_zar, axis=1)


In [23]:
test_data.head()

,user_id,measurement_date,hours_streaming,hours_social,hours_messaging,hours_gaming,is_peak_hour_user,is_weekend,age_group,plan_type,device_type,network_type,data_usage_category,streaming_data_gb,social_data_gb,messaging_data_gb,gaming_data_gb,total_data_gb,predicted_data_gb,prediction_error_gb,absolute_error_gb,high_predicted_usage,arpu_zar
0,SUB8984705,2026-03-31,1.73,4.73,1.23,0.85,1,1,25-34,Postpaid_Premium,Premium_Smartphone,4G,Moderate,2.17,1.38,0.01,0.11,4.13,3.92,0.22,0.22,False,357.09
1,SUB6140709,2026-03-20,0.40,0.26,0.17,1.82,0,0,55+,Prepaid_Monthly,Mid_Range,4G+,Light,0.55,0.07,0.00,0.29,1.01,0.83,0.18,0.18,False,109.11
2,SUB3654810,2026-04-16,0.91,0.28,0.01,0.03,0,0,55+,Prepaid_Daily,Mid_Range,4G,Light,1.00,0.05,0.00,0.00,1.04,1.14,-0.09,0.09,False,133.86
3,SUB6876924,2026-04-01,1.66,1.18,0.09,0.25,0,0,18-24,Postpaid_Premium,Basic_Phone,4G+,Moderate,3.95,0.29,0.00,0.03,3.88,4.65,-0.77,0.77,True,336.88
4,SUB3498002,2026-03-27,0.45,1.87,0.05,0.63,0,0,55+,Postpaid_Unlimited,Premium_Smartphone,4G,Light,0.79,0.36,0.00,0.12,1.33,1.12,0.21,0.21,False,134.93


In [22]:
# ============================================================
# ARPU AND PREDICTED USAGE BY SEGMENT
# ============================================================

segment_analysis = (
    test_data
    .groupby(["age_group", "device_type"])
    .agg(
        customers=("user_id", "count"),
        avg_actual_usage_gb=("total_data_gb", "mean"),
        avg_predicted_usage_gb=("predicted_data_gb", "mean"),
        avg_prediction_error_gb=("absolute_error_gb", "mean")
    )
    .reset_index()
)

segment_analysis = segment_analysis.sort_values(
    "avg_predicted_usage_gb",
    ascending=False
)

segment_analysis = segment_analysis.sort_values(
    "avg_arpu_zar",
    ascending=False
)

display(segment_analysis)

KeyError: 'avg_arpu_zar'

In [17]:
# ============================================================
# HIGH-VALUE + HIGH-USAGE CUSTOMERS
# ============================================================

usage_threshold = (
    test_data["predicted_data_gb"].quantile(0.75)
)

test_data["high_predicted_usage"] = (
    test_data["predicted_data_gb"]
    >= usage_threshold
)

target_customers = test_data[
    (
        test_data["high_value_customer"]
    )
    &
    (
        test_data["high_predicted_usage"]
    )
].copy()

print("=" * 70)
print("HIGH-VALUE / HIGH-PREDICTED-USAGE CUSTOMERS")
print("=" * 70)

print(
    f"Customers identified: "
    f"{len(target_customers):,}"
)

print(
    f"Average ARPU: "
    f"R{target_customers['arpu_zar'].mean():,.2f}"
)

print(
    f"Average predicted usage: "
    f"{target_customers['predicted_data_gb'].mean():,.2f} GB"
)

KeyError: 'high_value_customer'

In [18]:
# ============================================================
# PEAK-HOUR DEMAND
# ============================================================

peak_data = test_data[
    test_data["is_peak_hour_user"] == 1
]

off_peak_data = test_data[
    test_data["is_peak_hour_user"] == 0
]

peak_predicted_usage = (
    peak_data["predicted_data_gb"].sum()
)

off_peak_predicted_usage = (
    off_peak_data["predicted_data_gb"].sum()
)

total_predicted_usage = (
    test_data["predicted_data_gb"].sum()
)

peak_share = (
    peak_predicted_usage
    / total_predicted_usage
    * 100
)

print("=" * 70)
print("PEAK-HOUR DEMAND ANALYSIS")
print("=" * 70)

print(
    f"Peak predicted usage: "
    f"{peak_predicted_usage:,.2f} GB"
)

print(
    f"Off-peak predicted usage: "
    f"{off_peak_predicted_usage:,.2f} GB"
)

print(
    f"Peak-hour demand share: "
    f"{peak_share:.2f}%"
)

PEAK-HOUR DEMAND ANALYSIS
Peak predicted usage: 2,439.98 GB
Off-peak predicted usage: 4,396.84 GB
Peak-hour demand share: 35.69%
